# 01 — PySpark Fundamentals

Core concepts: SparkSession, DataFrame creation, basic operations.

## 0. Java Setup (PySpark requires Java)

In [ ]:
import os, shutil, subprocess, sys

def _find_java():
    """Check if java is available on PATH or in JAVA_HOME."""
    if os.environ.get("JAVA_HOME"):
        java_bin = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
        if os.path.isfile(java_bin):
            return True
    return shutil.which("java") is not None

def _find_installed_jdk():
    """Look for a previously installed JDK in ~/.jdk."""
    jdk_dir = os.path.expanduser("~/.jdk")
    if os.path.exists(jdk_dir):
        for d in sorted(os.listdir(jdk_dir)):
            candidate = os.path.join(jdk_dir, d)
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "bin", "java")):
                return candidate
    return None

# Auto-install Java if not available (required by PySpark)
if not _find_java():
    prev = _find_installed_jdk()
    if prev:
        os.environ["JAVA_HOME"] = prev
        print(f"Using JAVA_HOME={prev}")
    else:
        print("Java not found. Installing JDK 17 via install-jdk...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "install-jdk"])
        import jdk
        path = jdk.install("17")
        os.environ["JAVA_HOME"] = path
        print(f"JAVA_HOME set to {path}")
else:
    print(f"Java found. JAVA_HOME={os.environ.get('JAVA_HOME', '(system)')}")

## 1. Creating a SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Module12-Fundamentals") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print(f"App name: {spark.sparkContext.appName}")

## 2. Creating DataFrames

### From Python lists

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# From list of tuples (schema inferred)
data = [("Alice", 30, 95000.0), ("Bob", 25, 88000.0), ("Charlie", 35, 72000.0)]
df = spark.createDataFrame(data, ["name", "age", "salary"])
df.show()
df.printSchema()

### With explicit schema

In [ ]:
schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("age", IntegerType(), nullable=True),
    StructField("salary", DoubleType(), nullable=True),
])

df2 = spark.createDataFrame(data, schema)
df2.printSchema()

### From Pandas DataFrame

In [ ]:
import pandas as pd

pdf = pd.DataFrame({"product": ["Laptop", "Phone", "Tablet"], "price": [1200, 800, 500]})
sdf = spark.createDataFrame(pdf)
sdf.show()

### From CSV file

In [ ]:
import os

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

emp_df = spark.read.csv(os.path.join(FIXTURES, "employees.csv"), header=True, inferSchema=True)
emp_df.show(5)
emp_df.printSchema()
print(f"Rows: {emp_df.count()}, Columns: {len(emp_df.columns)}")

## 3. Basic DataFrame Operations

### select, filter, withColumn

In [ ]:
from pyspark.sql import functions as F

# select columns
emp_df.select("name", "department", "salary").show(5)

# filter rows
eng = emp_df.filter(F.col("department") == "Engineering")
eng.show()

# add/modify column
emp_df.withColumn("salary_k", F.round(F.col("salary") / 1000, 1)).select("name", "salary", "salary_k").show(5)

### orderBy, distinct, drop

In [ ]:
# sort
emp_df.orderBy(F.col("salary").desc()).select("name", "salary").show(5)

# distinct departments
emp_df.select("department").distinct().show()

# drop column
emp_df.drop("hire_date").columns

### Column expressions and when/otherwise

In [ ]:
emp_df.withColumn(
    "salary_band",
    F.when(F.col("salary") >= 95000, "Senior")
     .when(F.col("salary") >= 75000, "Mid")
     .otherwise("Junior")
).select("name", "salary", "salary_band").show()

## 4. String and Date Functions

In [ ]:
emp_df.select(
    "name",
    F.upper(F.col("name")).alias("name_upper"),
    F.length(F.col("name")).alias("name_len"),
    F.col("hire_date"),
    F.year(F.col("hire_date")).alias("hire_year"),
    F.datediff(F.current_date(), F.col("hire_date")).alias("days_employed"),
).show(5)

## 5. Handling Nulls

In [ ]:
data_nulls = [(1, "Alice", 100), (2, None, 200), (3, "Charlie", None)]
df_nulls = spark.createDataFrame(data_nulls, ["id", "name", "value"])

# Drop rows with any null
df_nulls.dropna().show()

# Fill nulls
df_nulls.fillna({"name": "Unknown", "value": 0}).show()

# Check for null
df_nulls.filter(F.col("name").isNull()).show()

## 6. Collecting Results

In [ ]:
# collect() — returns list of Row objects
rows = emp_df.select("name", "salary").limit(3).collect()
for row in rows:
    print(f"{row['name']}: ${row['salary']:,}")

# toPandas() — converts to pandas DataFrame
pdf = emp_df.select("name", "department", "salary").limit(5).toPandas()
print(type(pdf))
pdf

## Cleanup

In [ ]:
spark.stop()
print('SparkSession stopped.')